# 00 — Grouped validation audit

This notebook only loads and visualizes the frozen grouped-validation artifacts. It does not construct splits, train a model, select features, or access challenge test data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from qrt_forecasting.v2.folds import load_assignment_artifacts

In [ ]:
def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Unable to locate repository root.')


repo_root = find_repository_root(Path.cwd().resolve())
assignment_path = repo_root / 'artifacts' / 'folds' / 'v2_grouped_assignment.csv'
manifest_path = repo_root / 'reports' / 'validation' / 'v2_grouped_folds_manifest.json'
assignment, manifest = load_assignment_artifacts(assignment_path, manifest_path)

## Frozen artifact identity

In [ ]:
pd.Series({
    'selected_splitter': manifest['splitter_selection']['selected_splitter'],
    'data_fingerprint_sha256': manifest['data_fingerprint_sha256'],
    'assignment_sha256': manifest['assignment_sha256'],
    'assignment_rows': len(assignment),
})

## Structural comparison of splitter candidates

In [ ]:
audit_rows = []
for audit in manifest['splitter_selection']['candidate_audits']:
    audit_rows.append({
        'splitter': audit['splitter'],
        'valid': audit['valid'],
        'max_relative_row_deviation': audit['max_relative_row_deviation'],
        'max_relative_group_deviation': audit['max_relative_group_deviation'],
        'max_absolute_prevalence_deviation': audit['max_absolute_prevalence_deviation'],
        'worst_balance_deviation': audit['worst_balance_deviation'],
    })
splitter_audit = pd.DataFrame(audit_rows).set_index('splitter')
splitter_audit

In [ ]:
balance_columns = [
    'max_relative_row_deviation',
    'max_relative_group_deviation',
    'max_absolute_prevalence_deviation',
]
ax = splitter_audit[balance_columns].plot.bar(figsize=(10, 5))
ax.set_title('Maximum validation-fold imbalance by splitter')
ax.set_ylabel('Deviation')
ax.set_xlabel('')
plt.xticks(rotation=0)
plt.tight_layout()

## Development and frozen lockbox

In [ ]:
partition = pd.DataFrame({
    role: manifest['partition'][role]
    for role in ('development', 'lockbox')
}).T
partition[['n_rows', 'n_groups', 'positive_rate', 'row_fraction', 'group_fraction']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
partition['n_rows'].plot.bar(ax=axes[0], title='Rows by role')
partition['n_groups'].plot.bar(ax=axes[1], title='TS groups by role')
for axis in axes:
    axis.set_xlabel('')
    axis.tick_params(axis='x', rotation=0)
plt.tight_layout()

## Definitive development folds

In [ ]:
development_folds = pd.DataFrame(
    manifest['partition']['development_fold_audit']['folds']
).set_index('fold_id')
development_folds[['n_rows', 'n_groups', 'n_negative', 'n_positive', 'positive_rate']]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
development_folds['n_rows'].plot.bar(ax=axes[0], title='Rows per fold')
development_folds['n_groups'].plot.bar(ax=axes[1], title='TS groups per fold')
development_folds['positive_rate'].plot.bar(ax=axes[2], title='Positive rate per fold')
axes[2].axhline(
    manifest['partition']['development']['positive_rate'],
    color='black',
    linestyle='--',
    label='development',
)
axes[2].legend()
for axis in axes:
    axis.set_xlabel('fold_id')
    axis.tick_params(axis='x', rotation=0)
plt.tight_layout()

## Audit conclusion

The selected splitter satisfies the hard grouping and coverage invariants. The lockbox is frozen by complete `TS` groups and has no development fold ID. All model-development work must use only rows whose role is `development`.